# Network Configuration Testing with Batfish

This notebook demonstrates how to use [Batfish](https://www.batfish.org/) to analyze and validate network configurations **without needing physical devices**.

## Lab Topology
```
[192.168.1.0/24] -- router1 (10.0.12.1) ------ (10.0.12.2) router2 -- [192.168.2.0/24]
```

## Prerequisites
- Batfish service running: `docker run -d -p 9997:9997 -p 9996:9996 batfish/batfish`
- Python packages: `pip install pybatfish pandas`

## 1. Setup & Connection

In [ ]:
import os
import pandas as pd
from pybatfish.client.session import Session
from pybatfish.datamodel import HeaderConstraints, PathConstraints

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)

# Connect to Batfish (running locally via Docker)
BATFISH_HOST = os.getenv('BATFISH_HOST', 'localhost')
bf = Session(host=BATFISH_HOST)

print(f"Connected to Batfish at {BATFISH_HOST}")

Connected to Batfish at 192.168.1.240


## 2. Load Network Snapshot

A "snapshot" is a folder of router/switch configuration files that Batfish ingests and models.

In [44]:
SNAPSHOT_DIR = './snapshot'
NETWORK_NAME = 'demo-network'
SNAPSHOT_NAME = 'baseline'

bf.set_network(NETWORK_NAME)
bf.init_snapshot(SNAPSHOT_DIR, name=SNAPSHOT_NAME, overwrite=True)

print(f"Snapshot '{SNAPSHOT_NAME}' loaded successfully.")

Snapshot 'baseline' loaded successfully.


## 3. Inventory – What Did Batfish Parse?

Check what nodes and interfaces Batfish discovered from our configs.

In [45]:
# Node inventory
nodes = bf.q.nodeProperties().answer().frame()
print("=== Nodes ===")
print(nodes[['Node', 'Configuration_Format', 'NTP_Servers']].to_string(index=False))

=== Nodes ===
   Node Configuration_Format NTP_Servers
  host1                 HOST          []
  host2                 HOST          []
router1            CISCO_IOS          []
router2            CISCO_IOS          []


In [46]:
# Interface inventory
ifaces = bf.q.interfaceProperties().answer().frame()
print("=== Interfaces ===")
print(ifaces[['Interface', 'Primary_Address', 'Admin_Up', 'Active']].to_string(index=False))

=== Interfaces ===
                  Interface  Primary_Address Admin_Up Active
                host1[eth0] 192.168.1.100/24     True   True
                host2[eth0] 192.168.2.100/24     True   True
router1[GigabitEthernet0/0]     10.0.12.1/30     True   True
router1[GigabitEthernet0/1]   192.168.1.1/24     True   True
         router1[Loopback0]       1.1.1.1/32     True   True
router2[GigabitEthernet0/0]     10.0.12.2/30     True   True
router2[GigabitEthernet0/1]   192.168.2.1/24     True   True
         router2[Loopback0]       2.2.2.2/32     True   True


## 4. Test: Parse Warnings

Check if Batfish had trouble parsing any part of the configuration. Parse warnings can indicate unsupported syntax or configuration mistakes.

In [47]:
parse_warnings = bf.q.fileParseStatus().answer().frame()
print("=== File Parse Status ===")
print(parse_warnings[['File_Name', 'Status', 'File_Format', 'Nodes']].to_string(index=False))

# Fail if any file has FAILED parse status
failed = parse_warnings[parse_warnings['Status'] == 'FAILED']
assert len(failed) == 0, f"FAIL: {len(failed)} config file(s) failed to parse!\n{failed}"
print("\nPASS: All configuration files parsed successfully.")

=== File Parse Status ===
          File_Name Status File_Format       Nodes
configs/router1.cfg PASSED   CISCO_IOS ['router1']
configs/router2.cfg PASSED   CISCO_IOS ['router2']
   hosts/host1.json PASSED        HOST   ['host1']
   hosts/host2.json PASSED        HOST   ['host2']

PASS: All configuration files parsed successfully.


## 5. Test: Routing Table Reachability

Verify that both routers have routes to each other's internal subnets (end-to-end reachability via OSPF).

In [48]:
routes = bf.q.routes().answer().frame()
print("=== Routing Tables ===")
print(routes[['Node', 'VRF', 'Network', 'Next_Hop_IP', 'Protocol', 'Admin_Distance', 'Metric']].to_string(index=False))

=== Routing Tables ===
   Node     VRF          Network    Next_Hop_IP  Protocol Admin_Distance Metric
  host1 default        0.0.0.0/0    192.168.1.1    static              1      0
  host1 default   192.168.1.0/24 AUTO/NONE(-1l) connected              0      0
  host1 default 192.168.1.100/32 AUTO/NONE(-1l)     local              0      0
  host2 default        0.0.0.0/0    192.168.2.1    static              1      0
  host2 default   192.168.2.0/24 AUTO/NONE(-1l) connected              0      0
  host2 default 192.168.2.100/32 AUTO/NONE(-1l)     local              0      0
router1 default        0.0.0.0/0      10.0.12.2    static              1      0
router1 default       1.1.1.1/32 AUTO/NONE(-1l) connected              0      0
router1 default       2.2.2.2/32      10.0.12.2      ospf            110      2
router1 default     10.0.12.0/30 AUTO/NONE(-1l) connected              0      0
router1 default     10.0.12.1/32 AUTO/NONE(-1l)     local              0      0
router1 default  

In [49]:
# Verify router1 has a route to 192.168.2.0/24
r1_routes = routes[routes['Node'] == 'router1']
r1_has_route = r1_routes[r1_routes['Network'] == '192.168.2.0/24']
assert len(r1_has_route) > 0, "FAIL: router1 has no route to 192.168.2.0/24!"
print(f"PASS: router1 has route to 192.168.2.0/24 via {r1_has_route.iloc[0]['Next_Hop_IP']}")

# Verify router2 has a route to 192.168.1.0/24
r2_routes = routes[routes['Node'] == 'router2']
r2_has_route = r2_routes[r2_routes['Network'] == '192.168.1.0/24']
assert len(r2_has_route) > 0, "FAIL: router2 has no route to 192.168.1.0/24!"
print(f"PASS: router2 has route to 192.168.1.0/24 via {r2_has_route.iloc[0]['Next_Hop_IP']}")

PASS: router1 has route to 192.168.2.0/24 via 10.0.12.2
PASS: router2 has route to 192.168.1.0/24 via 10.0.12.1


## 6. Test: Traceroute / Reachability

Simulate a packet traversal from 192.168.1.100 to 192.168.2.100 and verify it is `ACCEPTED`.

In [50]:
traceroute_result = bf.q.traceroute(
    startLocation='router1[GigabitEthernet0/1]',
    headers=HeaderConstraints(
        srcIps='192.168.1.100',
        dstIps='192.168.2.100',
        ipProtocols=['TCP'],
        dstPorts='80'
    )
).answer().frame()

print("=== Traceroute: 192.168.1.100 -> 192.168.2.100 (HTTP) ===")
for _, row in traceroute_result.iterrows():
    print(f"Flow: {row['Flow']}")
    for trace in row['Traces']:
        print(f"  Disposition: {trace.disposition}")
        for hop in trace.hops:
            print(f"  -> {hop.node}")

=== Traceroute: 192.168.1.100 -> 192.168.2.100 (HTTP) ===
Flow: start=router1 [192.168.1.100:49152->192.168.2.100:80 TCP (SYN)]
  Disposition: ACCEPTED
  -> router1
  -> router2
  -> host2


In [51]:
# Assert all traces are ACCEPTED
for _, row in traceroute_result.iterrows():
    for trace in row['Traces']:
        assert trace.disposition == 'ACCEPTED', \
            f"FAIL: Traffic was {trace.disposition}, expected ACCEPTED!"

print("PASS: HTTP traffic from 192.168.1.0/24 to 192.168.2.0/24 is reachable.")

PASS: HTTP traffic from 192.168.1.0/24 to 192.168.2.0/24 is reachable.


## 7. Test: Security Policy – Telnet Must Be Blocked

We have ACLs configured to block Telnet (TCP/23). Verify this is enforced.

In [53]:
telnet_trace = bf.q.traceroute(
    startLocation='router1[GigabitEthernet0/0]',
    headers=HeaderConstraints(
        srcIps='192.168.1.100',
        dstIps='192.168.2.0/24',
        ipProtocols=['TCP'],
        dstPorts='23'   # Telnet
    )
).answer().frame()

print("=== Traceroute: External -> 192.168.1.0/24 (TELNET) ===")
for _, row in telnet_trace.iterrows():
    for trace in row['Traces']:
        print(f"  Disposition: {trace.disposition}")

=== Traceroute: External -> 192.168.1.0/24 (TELNET) ===
  Disposition: DENIED_IN


In [54]:
# Assert telnet is DENIED
for _, row in telnet_trace.iterrows():
    for trace in row['Traces']:
        assert trace.disposition in ('DENIED_IN', 'DENIED_OUT'), \
            f"FAIL: Telnet was {trace.disposition}, expected DENIED!"

print("PASS: Telnet (TCP/23) is correctly blocked by ACL.")

PASS: Telnet (TCP/23) is correctly blocked by ACL.


## 8. Test: ACL Audit – Check ACL Rules

Get a full breakdown of what each ACL permits and denies.

In [60]:
acls = bf.q.namedStructures(structureTypes='IP_ACCESS_LIST').answer().frame()
print("=== ACL Definitions ===")
print(acls[['Node', 'Structure_Type', 'Structure_Name']].to_string(index=False))

=== ACL Definitions ===
   Node Structure_Type Structure_Name
router1 IP_Access_List   BLOCK_TELNET
router2 IP_Access_List   BLOCK_TELNET


In [61]:
# Test specific flow against an ACL
acl_test = bf.q.testFilters(
    filters='BLOCK_TELNET',
    nodes='router1',
    headers=HeaderConstraints(
        srcIps='0.0.0.0/0',
        dstIps='0.0.0.0/0',
        ipProtocols=['TCP'],
        dstPorts='23'
    )
).answer().frame()

print("=== ACL Test: Telnet against BLOCK_TELNET ===")
print(acl_test[['Node', 'Filter_Name', 'Flow', 'Action', 'Line_Content']].to_string(index=False))

assert all(acl_test['Action'] == 'DENY'), "FAIL: ACL should deny telnet!"
print("\nPASS: BLOCK_TELNET ACL correctly denies Telnet traffic.")

=== ACL Test: Telnet against BLOCK_TELNET ===
   Node  Filter_Name                                                 Flow Action           Line_Content
router1 BLOCK_TELNET start=router1 [10.0.0.0:49152->8.8.8.8:23 TCP (SYN)]   DENY deny tcp any any eq 23

PASS: BLOCK_TELNET ACL correctly denies Telnet traffic.


## 9. Test: Undefined References

Find references to structures (ACLs, route-maps, etc.) that are used but never defined — a common misconfiguration.

In [62]:
undefined_refs = bf.q.undefinedReferences().answer().frame()

if len(undefined_refs) == 0:
    print("PASS: No undefined references found.")
else:
    print("FAIL: Undefined references detected!")
    print(undefined_refs.to_string(index=False))
    raise AssertionError(f"{len(undefined_refs)} undefined reference(s) found!")

PASS: No undefined references found.


## 10. Test: Unused Structures

Identify defined structures (ACLs, prefix-lists, etc.) that are never referenced — helps keep configs clean.

In [63]:
unused = bf.q.unusedStructures().answer().frame()

if len(unused) == 0:
    print("PASS: No unused structures found.")
else:
    print(f"WARNING: {len(unused)} unused structure(s) found (not necessarily an error):")
    print(unused[['Node', 'Structure_Type', 'Structure_Name']].to_string(index=False))

PASS: No unused structures found.


## 11. Summary

Run all tests and produce a pass/fail summary.

In [64]:
test_results = {
    'Config parse (no failures)': True,
    'router1 route to 192.168.2.0/24': len(r1_has_route) > 0,
    'router2 route to 192.168.1.0/24': len(r2_has_route) > 0,
    'HTTP reachability 192.168.1.x -> 192.168.2.x': all(
        trace.disposition == 'ACCEPTED'
        for _, row in traceroute_result.iterrows()
        for trace in row['Traces']
    ),
    'Telnet blocked by ACL': all(
        trace.disposition in ('DENIED_IN', 'DENIED_OUT')
        for _, row in telnet_trace.iterrows()
        for trace in row['Traces']
    ),
    'No undefined references': len(undefined_refs) == 0,
}

print("=" * 50)
print("        BATFISH TEST SUMMARY")
print("=" * 50)
all_passed = True
for test, result in test_results.items():
    status = 'PASS' if result else 'FAIL'
    if not result:
        all_passed = False
    print(f"  [{status}]  {test}")
print("=" * 50)
if all_passed:
    print("  ALL TESTS PASSED")
else:
    print("  SOME TESTS FAILED")
print("=" * 50)

        BATFISH TEST SUMMARY
  [PASS]  Config parse (no failures)
  [PASS]  router1 route to 192.168.2.0/24
  [PASS]  router2 route to 192.168.1.0/24
  [PASS]  HTTP reachability 192.168.1.x -> 192.168.2.x
  [PASS]  Telnet blocked by ACL
  [PASS]  No undefined references
  ALL TESTS PASSED
